In [ ]:
words = open('../data/makemore/names.txt', 'r').read().splitlines()
words[:10]

In [ ]:
len(words)

In [ ]:
min(len(w) for w in words)

In [ ]:
max(len(w) for w in words)

# 1 Bigrams

## 1.1 Examples

In [ ]:
# [Example] Iterate over the pairs of characters in the list of words
for w in words[:3]:
    chs = ['<S>'] + list(w) + ['<E>']  # Add special <S> (start) and <E> (end) characters
    for ch1, ch2 in zip(chs, chs[1:]):
        print(ch1, ch2)

In [ ]:
# [Example] Create a bigram as a dictionary
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']  # Add special <S> (start) and <E> (end) characters
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1
        # print(ch1, ch2)

sorted(b.items(), key=lambda kv: -kv[1])

## 1.2 Train the bigram

In [ ]:
import torch

In [ ]:
N = torch.ones((27, 27), dtype=torch.int32)  # Not zeros, to help calculating the loss bellow (to avoid inf for the log_likelihood)

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

In [ ]:
# Create a bigram as a Torch Tensor (array)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16, 16))
plt.imshow(N, cmap='Blues')
for i in range(len(itos)):
    for j in range(len(itos)):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha='center', va='bottom', color='gray')
        plt.text(j, i, N[i, j].item(), ha='center', va='top', color='gray')
plt.axis('off')

The N is an array containing everything we need for the Bigram model. We can sample characters from this array based on the probability distribution.

## 1.3 Sample from the trained model

In [ ]:
N[0]

In [ ]:
p = N[0].float()
p /= p.sum()
p

In [ ]:
# Draw a first character based on the character's probability extracted from the 0-th row of the Bigram
g = torch.Generator().manual_seed(2147483647)
idx = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
itos[idx]

In [ ]:
# g = torch.Generator().manual_seed(2147483647)
# p = torch.rand(3, generator=g)
# p /= p.sum()
# p

In [ ]:
# torch.multinomial(p, num_samples=100, replacement=True, generator=g)

In [ ]:
P = N.float()
P /= P.sum(1, keepdim=True)

P[0].sum(), P[:, 0].sum()

In [ ]:
g = torch.Generator().manual_seed(2147483647)

out_words = []
for _ in range(5):
    out = []
    idx = 0
    while True:
        p = P[idx].float()

        # [Not efficient, replaced] Probabilities from the trained model (Bigrams)
        # p = N[idx].float()
        # p /= p.sum()

        # [Not efficient, replaced] Probabilities from the untrained model (all probs. are equal)
        # p = torch.ones(27) / 27.0

        idx = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[idx])
        if idx == 0:
            break

    gen_word = ''.join(out)
    out_words.append(gen_word)
    print(''.join(gen_word))

## 1.4 Bigrams model evaluation

In [ ]:
# [Example] Use negative log likelihood as a loss function (based on the character pairs probs)
# - Probs should be multiplied (likelihood), but the product will be tiny
# - We use log (log likelihood), bacuse it is monotonic: closer to 0 - better, lower that 0 - worse
# - log(a*b*c) = log(a) + log(b) + log(c) We can use sum
# - To make it loss like - invert the number (negative log likelihood)
# - The final loss is calculated across all the words as an average value
# GOAL: maximize the likelihood of the data w.r.t. the model parameters (statistical modeling),
# which is an equivalent of minimizing the average negative log likelihood (it is just scaling)
log_likelihood = 0.0
n_count = 0
for w in words:  # Set here to [:3] to look at the example
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]

        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n_count += 1
        # print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')  # Uncomment here to look at the example

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n_count}')

Above is the loss calculated for the training set

In [ ]:
log_likelihood = 0.0
n_count = 0
for w in out_words:  # Set here to [:3] to look at the example
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]

        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n_count += 1
        # print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')  # Uncomment here to look at the example

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n_count}')

Above is an exmaple of the loss calculated for the generated words by the statistically trained Bigram model.

The problem is **the current model doesn't have any parameters we can change so we cannot improve the model**.